# 02 --- Wideband Channelizer

Channelization splits a **wideband IQ stream** into multiple narrowband
channels, each decimated to a lower sample rate. This is the first
processing stage in the PINGU pipeline after IQ capture.

PINGU provides two channelizer implementations:

1. **PolyphaseChannelizer** -- polyphase filterbank analysis (primary).
2. **FFTChannelizer** -- FFT-based overlap-save method (reference).

This notebook demonstrates both on a synthetic multi-tone wideband signal.

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from pingu.channelizer.polyphase import PolyphaseChannelizer
from pingu.channelizer.fft import FFTChannelizer
from pingu.types import IQFrame

# Reproducible random generator
rng = np.random.default_rng(seed=42)

print("Imports OK")

## Create a Wideband Test Signal

We create a synthetic wideband IQ frame containing **3 tones** at known
frequency offsets from the centre frequency. This lets us verify that the
channelizer places each tone in the correct output channel.

| Tone | Offset (Hz) | Description               |
|------|-------------|---------------------------|
| 1    | +2 000      | Near centre               |
| 2    | +8 000      | Upper portion of band     |
| 3    | -5 000      | Lower portion of band     |

In [ ]:
# Parameters
SAMPLE_RATE = 48_000      # Hz
CENTER_FREQ = 14.1e6      # Hz (20-m amateur band)
DURATION = 0.05           # seconds
N_SAMPLES = int(SAMPLE_RATE * DURATION)

# Time vector
t = np.arange(N_SAMPLES, dtype=np.float64) / SAMPLE_RATE

# Build wideband signal: 3 tones at different offsets + light noise floor
tone_offsets = [+2000, +8000, -5000]  # Hz relative to center
signal = np.zeros(N_SAMPLES, dtype=np.complex128)
for offset in tone_offsets:
    signal += np.exp(2j * np.pi * offset * t)

# Add a light noise floor
noise = (rng.standard_normal(N_SAMPLES) + 1j * rng.standard_normal(N_SAMPLES)) / np.sqrt(2)
signal += 0.1 * noise
signal = signal.astype(np.complex64)

# Wrap in an IQFrame
frame = IQFrame(
    receiver_id="RX-DEMO",
    samples=signal,
    sample_rate=SAMPLE_RATE,
    center_freq=CENTER_FREQ,
    timestamp=0.0,
)

print(f"IQFrame: {len(frame.samples)} samples, fs={frame.sample_rate} Hz, fc={frame.center_freq/1e6:.3f} MHz")

# Spectrogram of the input signal
fig, ax = plt.subplots(figsize=(12, 4))
ax.specgram(signal, NFFT=256, Fs=SAMPLE_RATE, noverlap=128, cmap="viridis")
ax.set_title("Input Wideband Spectrogram", fontsize=13, fontweight="bold")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Frequency (Hz)")
fig.tight_layout()
plt.show()

## Polyphase Filterbank Channelizer

The `PolyphaseChannelizer` implements an M-path polyphase analysis
filterbank. A prototype lowpass filter is decomposed into M branches, and
each input block is filtered and transformed via an M-point FFT to extract
the narrowband channel outputs.

In [ ]:
# Channelize with polyphase filterbank
N_CHANNELS = 16
poly_channelizer = PolyphaseChannelizer(n_channels=N_CHANNELS)
poly_result = poly_channelizer.channelize(frame)

print(f"Polyphase output shape : {poly_result.channels.shape}  (n_channels, n_samples)")
print(f"Channel bandwidth      : {poly_result.channel_bw:.0f} Hz")
print(f"Per-channel sample rate: {poly_result.sample_rate:.0f} Hz")

# Compute energy per channel
poly_energy = np.sum(np.abs(poly_result.channels.astype(np.complex128)) ** 2, axis=1)
poly_energy_db = 10 * np.log10(poly_energy + 1e-30)

# Bar chart of energy per channel
fig, ax = plt.subplots(figsize=(12, 5))
channel_labels = [f"Ch {i}" for i in range(N_CHANNELS)]
bars = ax.bar(range(N_CHANNELS), poly_energy_db, color="steelblue", edgecolor="navy", alpha=0.85)
ax.set_xticks(range(N_CHANNELS))
ax.set_xticklabels(channel_labels, rotation=45)
ax.set_xlabel("Channel Index")
ax.set_ylabel("Energy (dB)")
ax.set_title("Polyphase Channelizer -- Energy per Channel", fontsize=13, fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")
fig.tight_layout()
plt.show()

# Report which channels have the highest energy (should correspond to tone offsets)
peak_channels = np.argsort(poly_energy)[-3:][::-1]
print("\nTop-3 channels by energy (polyphase):")
for ch in peak_channels:
    freq_offset = poly_result.channel_freqs[ch] - CENTER_FREQ
    print(f"  Channel {ch:2d}  energy={poly_energy_db[ch]:.1f} dB  freq_offset={freq_offset:+.0f} Hz")

In [ ]:
# Plot the time-domain output of the 3 channels containing the tones
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

for idx, ch in enumerate(peak_channels):
    ax = axes[idx]
    ch_samples = poly_result.channels[ch]
    ch_time_ms = np.arange(len(ch_samples)) / poly_result.sample_rate * 1000
    ax.plot(ch_time_ms, np.real(ch_samples), linewidth=0.8, label="Real")
    ax.plot(ch_time_ms, np.imag(ch_samples), linewidth=0.8, alpha=0.6, label="Imag")
    freq_offset = poly_result.channel_freqs[ch] - CENTER_FREQ
    ax.set_title(f"Channel {ch} (offset {freq_offset:+.0f} Hz)", fontsize=11, fontweight="bold")
    ax.set_ylabel("Amplitude")
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time (ms)")
fig.suptitle("Polyphase -- Active Channel Time-Domain Outputs", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

## FFT Channelizer (Reference)

The `FFTChannelizer` uses an FFT-based overlap-save method. It is simpler
but less spectrally efficient than the polyphase approach. We use it here as
a reference to cross-check the polyphase results.

In [ ]:
# Channelize with FFT method
fft_channelizer = FFTChannelizer(n_channels=N_CHANNELS)
fft_result = fft_channelizer.channelize(frame)

print(f"FFT output shape       : {fft_result.channels.shape}  (n_channels, n_samples)")
print(f"Channel bandwidth      : {fft_result.channel_bw:.0f} Hz")
print(f"Per-channel sample rate: {fft_result.sample_rate:.0f} Hz")

# Compute energy per channel
fft_energy = np.sum(np.abs(fft_result.channels.astype(np.complex128)) ** 2, axis=1)
fft_energy_db = 10 * np.log10(fft_energy + 1e-30)

# Side-by-side comparison bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

ax1.bar(range(N_CHANNELS), poly_energy_db, color="steelblue", edgecolor="navy", alpha=0.85)
ax1.set_xticks(range(N_CHANNELS))
ax1.set_xticklabels(channel_labels, rotation=45)
ax1.set_xlabel("Channel Index")
ax1.set_ylabel("Energy (dB)")
ax1.set_title("Polyphase", fontsize=12, fontweight="bold")
ax1.grid(True, alpha=0.3, axis="y")

ax2.bar(range(N_CHANNELS), fft_energy_db, color="coral", edgecolor="darkred", alpha=0.85)
ax2.set_xticks(range(N_CHANNELS))
ax2.set_xticklabels(channel_labels, rotation=45)
ax2.set_xlabel("Channel Index")
ax2.set_title("FFT", fontsize=12, fontweight="bold")
ax2.grid(True, alpha=0.3, axis="y")

fig.suptitle("Energy per Channel -- Polyphase vs FFT", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

# Report top channels for FFT
fft_peak_channels = np.argsort(fft_energy)[-3:][::-1]
print("\nTop-3 channels by energy (FFT):")
for ch in fft_peak_channels:
    freq_offset = fft_result.channel_freqs[ch] - CENTER_FREQ
    print(f"  Channel {ch:2d}  energy={fft_energy_db[ch]:.1f} dB  freq_offset={freq_offset:+.0f} Hz")

## Energy Conservation

A well-designed channelizer should (approximately) conserve total signal
energy across its output channels. We verify this for both implementations.

In [ ]:
# Compute input energy
input_energy = float(np.sum(np.abs(frame.samples.astype(np.complex128)) ** 2))

# Polyphase total channel energy
poly_total_energy = float(np.sum(poly_energy))
poly_ratio = poly_total_energy / input_energy

# FFT total channel energy
fft_total_energy = float(np.sum(fft_energy))
fft_ratio = fft_total_energy / input_energy

print(f"Input energy          : {input_energy:.2f}")
print(f"Polyphase total energy: {poly_total_energy:.2f}  (ratio = {poly_ratio:.4f})")
print(f"FFT total energy      : {fft_total_energy:.2f}  (ratio = {fft_ratio:.4f})")
print()
print(f"Polyphase energy conservation: {poly_ratio * 100:.1f}%")
print(f"FFT energy conservation      : {fft_ratio * 100:.1f}%")

## Summary

- The **PolyphaseChannelizer** is the primary channelization method. It
  offers good spectral isolation between channels thanks to the prototype
  lowpass filter.
- The **FFTChannelizer** is a simpler reference implementation using
  windowed FFT segments.
- Both methods correctly identify the active channels corresponding to the
  injected tones.
- Energy conservation ratios confirm that signal power is preserved through
  the channelization process.